In [1]:
import re
from ntr.bhandar import dict_value
from collections import OrderedDict

In [1]:
import re
import sys
from ntr.bhandar import replace_longest, LOANWORD_MAP

def nepali_schwa_deletion(roman_word, original_devanagari_word):
    """
    Standard Nepali schwa deletion with exceptions.
    - Delete final 'a' unless:
        * Word length == 1 (e.g., 'ma', 'ta')
        * Original word ends with a vowel sign or modifier (ा,ि,ी,ु,ू,े,ै,ो,ौ,ं,ः)
        * Word is in the exception list (e.g., 'matra')
    """
    # Exception list for words that must keep final 'a'
    exceptions = {'matra', 'yatha', 'tatha', 'prakriti', 'kripa'}  # add as needed
    if roman_word in exceptions:
        return roman_word
    
    if len(roman_word) == 1:
        return roman_word
    
    # Check original Devanagari ending
    if original_devanagari_word and original_devanagari_word[-1] in 'ािीुूेैोौंः':
        return roman_word
    
    # If roman_word ends with 'a', remove it
    if roman_word.endswith('a'):
        return roman_word[:-1]
    return roman_word

def nep_to_rom(text):
    """Convert Devanagari text to romanised Nepali (lowercase, human typing style)."""
    # Split into words and newlines
    tokens = re.findall(r'\S+|\n', text)
    output_tokens = []
    
    for token in tokens:
        if token == '\n':
            output_tokens.append('\n')
            continue
        
        # Check loanword exception FIRST (exact match)
        if token in LOANWORD_MAP:
            output_tokens.append(LOANWORD_MAP[token])
            continue
        
        # Apply longest-match romanisation
        roman = replace_longest(token)
        
        # Remove any leftover halant
        roman = roman.replace('्', '')
        
        # Apply schwa deletion
        roman = nepali_schwa_deletion(roman, token)
        
        output_tokens.append(roman)
    
    result = ' '.join(output_tokens).replace(' \n', '\n').strip()

    return result

def convert_file(input_path, output_path):
    """Read Devanagari from input_path, write romanised text to output_path."""
    with open(input_path, 'r', encoding='utf-8') as f:
        nepali_text = f.read()
    roman_text = nep_to_rom(nepali_text)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(roman_text)
    print(f"Conversion complete. Output: {output_path}")



if __name__ == '__main__':
    convert_file('devnagari.txt', 'output.txt')

Conversion complete. Output: output.txt
